# TBsim demonstration in Google Colab

TBsim is an agent-based TB modelling framework built on Starsim. It enables fast, stochastic simulations of TB transmission, progression, and interventions using a modular API. The LSHTM spectrum-of-disease natural history is represented as: susceptible -> infection -> non-infectious -> asymptomatic -> symptomatic -> treatment/cleared/dead. This notebook demonstrates how to configure TBsim runs, apply care cascades, introduce prevention measures, and compare scenarios in a Colab-friendly setup.

## Research basis (summary and parameter table)

The draft research document describes Starsim/TBsim capabilities and design considerations. The following notes are derived to illustrate how one might set up TB/TB-HIV and related modules in TBsim. Values that are not explicitly provided in the document are noted as placeholders or TBsim defaults. The parameter table lists value, units, and source.

Parameter table (values are organized for clarity; placeholders are clearly labeled):

| Parameter | Value | Units | Source |
|---|---:|:---:|---|
| n_agents | 3000 | agents | research doc (baseline placeholder) |
| start | 2000-01-01 | date | TBsim default |
| stop | 2020-12-31 | date | TBsim default |
| dt | 7 | days | TBsim default |
| rand_seed | 1 | integer | TBsim default |
| beta | 0.2 | per year | research doc (beta per year placeholder) |
| init_prev | 0.05 | proportion | research doc (initial prevalence placeholder) |
| trans_asymp | 0.0 | ratio | TBsim default |
| rr_reinfection_rec | 1.0 | multiplier | TBsim default |
| rr_reinfection_treat | 1.0 | multiplier | TBsim default |
| networks | RandomNet(n_contacts=Poisson(5)) | network | TBsim default |
| analyzers | [DwtAnalyzer(scenario_name='baseline')] | analyzer | TBsim default |

Notes:
- Exact numeric values for many TB progression parameters are not supplied in the document; placeholders are marked. Users should replace placeholders with their calibrated values or TBsim defaults as appropriate for their study.
- The document emphasizes modular interaction between disease modules (e.g., HIV, malnutrition) and interventions; this notebook demonstrates structure without reimplementing TB models.

Table of parameters used in this notebook (value, units, source):

- n_agents: 3000, agents, research doc placeholder
- start: 2000-01-01, date, TBsim default
- stop: 2020-12-31, date, TBsim default
- dt: 7 days, time step, TBsim default
- rand_seed: 1, integer, TBsim default
- beta: 0.2, per year, research doc placeholder
- init_prev: 0.05, proportion, research doc placeholder
- trans_asymp: 0.0, ratio, TBsim default
- rr_reinfection_rec: 1.0, multiplier, TBsim default
- rr_reinfection_treat: 1.0, multiplier, TBsim default
- networks: RandomNet(n_contacts=poisson(lam=5)), TBsim default net
- analyzers: DwtAnalyzer(scenario_name='baseline'), TBsim default

In [ ]:
# 3. Setup
# Install TBsim from GitHub (ensures compatibility with Starsim 3.7)
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sciris as sc

# Install TBsim package (GitHub source as in API reference)
!pip -q install -q git+https://github.com/starsimhub/tbsim.git
import tbsim
import starsim as ss

print('TBsim version:', tbsim.__version__)
print('Starsim version:', ss.__version__)

In [ ]:
# 4. Shared parameters (BASE)
# Build a base parameter set from the research notes and TBsim defaults
BASE = dict(
    n_agents=3000,
    start=ss.date('2000-01-01'),
    stop=ss.date('2020-12-31'),
    dt=ss.days(7),
    rand_seed=1,
    verbose=0,
    beta=ss.peryear(0.2),
    init_prev=ss.bernoulli(0.05)
)

In [ ]:
# 5. Baseline natural history (code + Markdown)
# Baseline TB natural history with default TB model (no extra interventions)
import tbsim
import starsim as ss
# Construct a minimal baseline TB sim using a RandomNet and default TB model
networks = [ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=5), dur=0))]

baselinesim = tbsim.Sim(**BASE, networks=networks, analyzers=[tbsim.DwtAnalyzer(scenario_name='baseline')])
baselinesim.run()
tb = baselinesim.get_tb()
timevec = baselinesim.timevec

prevalence = baselinesim.results.flatten().tb_prevalence.values if hasattr(baselinesim.results.flatten(), 'tb_prevalence') else None
incidence = baselinesim.results.flatten().tb_incidence_kpy.values if hasattr(baselinesim.results.flatten(), 'tb_incidence_kpy') else None

print('Baseline TB model run completed.')

if prevalence is not None:
    plt.figure(figsize=(8,4))
    plt.plot(timevec, prevalence, label='Tb Prevalence')
    plt.xlabel('Time')
    plt.ylabel('Prevalence')
    plt.legend()
    plt.title('Baseline TB Prevalence over time')
    plt.grid(True)
    plt.show()

if incidence is not None:
    plt.figure(figsize=(8,4))
    plt.plot(timevec, incidence, label='Tb Incidence')
    plt.xlabel('Time')
    plt.ylabel('Incidence')
    plt.legend()
    plt.title('Baseline TB Incidence over time')
    plt.grid(True)
    plt.show()

In [ ]:
# 6. Care cascade (code + Markdown)
# Demonstrate care cascade: health-seeking -> screening -> confirmatory test -> treatment
health_seeking = tbsim.HealthSeekingBehavior(pars=dict(initial_care_seeking_rate=ss.perday(0.1)))
screen = tbsim.DxDelivery(name='screen', product=tbsim.CAD(), coverage=0.9, result_state='screen_positive')
confirm = tbsim.DxDelivery(name='confirm', product=tbsim.Xpert(), coverage=0.8, eligibility=lambda sim: sim.people.screen.screen_positive.uids if hasattr(sim, 'people') else None, result_state='diagnosed')
treatment = tbsim.TxDelivery(product=tbsim.DOTS())

 cascade_sim = tbsim.Sim(**BASE, networks=networks, interventions=[health_seeking, screen, confirm, treatment], analyzers=[tbsim.DwtAnalyzer(scenario_name='cascade')])
 cascade_sim.run()
 print('Cascade run completed.')

In [ ]:
# 7. Prevention (code + Markdown)
# Include BCG, household TPT (with a simple household network), and a BetaByYear transmission change
bcg = tbsim.BCGRoutine(pars=dict(coverage=0.8, age_range=[0,5]))
tpt = tbsim.TPTHousehold(pars=dict(coverage=0.8))
beta_by_year = tbsim.BetaByYear(pars=dict(years=[2010], x_beta=[0.7]))

prevention_sim = tbsim.Sim(**BASE, networks=networks, interventions=[bcg, tpt, beta_by_year], analyzers=[tbsim.DwtAnalyzer(scenario_name='prevention')])
prevention_sim.run()
print('Prevention scenario completed.')

In [ ]:
# 8. Comorbidities (code + Markdown)
# HIV and malnutrition connectors (demonstration only; not a full clinical calibration)
hiv = tbsim.HIV(pars=dict(init_prev=ss.bernoulli(0.0), init_onart=ss.bernoulli(0.0)))
hivint = tbsim.HivInterventions(pars=dict(use_prevalence=True, use_art=True, prevalence=0.2, percent_on_ART=0.5, min_age=15, max_age=60, start=ss.date('2005-01-01'), stop=ss.date('2020-12-31')))
nutrition = tbsim.Malnutrition()
sim_comorb = tbsim.Sim(**BASE, diseases=[hiv, nutrition], interventions=[hivint], connectors=[tbsim.TB_HIV_Connector(), tbsim.TB_Nutrition_Connector()])
sim_comorb.run()
print('Comorbidity scenario run.')

In [ ]:
# 9. Drug resistance (code + Markdown)
# Demonstrate TB resistance module and resistance statistics (note: a minimal demonstration)
tbr = tbsim.TBResistant(drugs=['RIF'], rel_fitness={'RIF': 0.9}, beta=ss.permonth(0.35), init_prev=ss.bernoulli(0.12))
tx = tbsim.TxDeliveryR(rate_sym=ss.peryear(1.0), product=tbsim.TxR(strains=tbr.strains, base_efficacy=0.8, resist_penalty={'RIF': 0.2}))
dr_sim = tbsim.Sim(tb_model=tbr, n_agents=2000, networks=ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=8), dur=0)), interventions=[tx], analyzers=[tbsim.ResistanceStats()], dt=ss.days(30), start=ss.date('2000-01-01'), stop=ss.date('2015-01-01'), verbose=0)
dr_sim.run()
print('Drug resistance scenario completed.')

In [ ]:
# 10. Scenario comparison (code)
# For demonstration we compare Baseline vs Cascade vs Prevention using simple summaries
results = {}
for name, sim in [('Baseline', baselinesim), ('Cascade', cascade_sim), ('Prevention', prevention_sim)]:
    if hasattr(sim, 'results'):
        r = sim.results.flatten()
        p = getattr(r, 'tb_prevalence', None)
        cum_inf = getattr(r, 'tb_cum_infections', None)
        deaths = getattr(r, 'tb_deaths_ppy', None)
        results[name] = {
            'prevalence_last': p.values[-1] if p is not None else None,
            'cum_infections': cum_inf.values[-1] if cum_inf is not None else None,
            'deaths_per_year': deaths.values[-1] if deaths is not None else None
        }
    else:
        results[name] = {'prevalence_last': None, 'cum_infections': None, 'deaths_per_year': None}

summary = pd.DataFrame.from_dict(results, orient='index')
print('Scenario comparison summary:')
print(summary)

plt.figure(figsize=(12,4))
for name, sim in [('Baseline', baselinesim), ('Cascade', cascade_sim), ('Prevention', prevention_sim)]:
    if hasattr(sim, 'results'):
        r = sim.results.flatten()
        p = getattr(r, 'tb_prevalence', None)
        if p is not None:
            plt.plot(r.yearvec.values, p.values, label=name)
plt.xlabel('Year')
plt.ylabel('Prevalence')
plt.legend()
plt.title('Prevalence over time across scenarios (baseline displays as line)')
plt.show()

In [ ]:
# 11. Dwell-time analysis (code)
# Use DwtAnalyzer results from cascade run (example)
if hasattr(cascade_sim, 'results'):
    dwt = cascade_sim.results.dwt if hasattr(cascade_sim.results, 'dwt') else None
    if dwt is not None:
        print('Dwell-time analyzer results available for cascade scenario.')
    else:
        print('Dwell-time analyzer results not extracted in this simple demonstration.')
else:
    print('Cascade sim results not available for dwell-time extraction.')

In [ ]:
# 12. Sanity checks (code)
# Basic assertions to verify outputs are finite and sensible
def _assert_non_negative(x, name):
    assert np.all(np.isfinite(x)), f'{name} contains non-finite values'
    assert np.all(x >= 0), f'{name} contains negative values'

for sim_name, sim in [('Baseline', baselinesim), ('Cascade', cascade_sim), ('Prevention', prevention_sim)]:
    if hasattr(sim, 'results'):
        r = sim.results.flatten()
        if hasattr(r, 'tb_prevalence'):
            _assert_non_negative(r.tb_prevalence.values, f'{sim_name} tb_prevalence')
        if hasattr(r, 'tb_incidence_kpy'):
            _assert_non_negative(r.tb_incidence_kpy.values, f'{sim_name} tb_incidence_kpy')
    else:
        print(f'No results for {sim_name} to sanity-check.')
print('Sanity checks completed.')

## Interpretation and limitations

- This notebook demonstrates the TBsim/Starsim API usage to assemble a baseline TB natural history, a care cascade, prevention measures, comorbidities, and drug resistance in a modular way. It follows patterns from the API reference but uses placeholder values where the research document does not provide concrete numbers. Some components (e.g., HIV/nutrition connectors and resistance statistics) are demonstrated at a structural level and may require calibration with empirical data for a real analysis.
- Outputs are illustrative. The TB progression, transmission, and intervention effects should be calibrated to real data before any policy guidance.
- Sections labeled as placeholders should be replaced with the appropriate calibrated inputs from the research dataset when available.